# Time-Based Routine Pattern Extraction

This notebook demonstrates how household routines can be automatically discovered from historical smart-home events.

Example pattern:

"living_room_light usually turns ON around 19:00"

The algorithm works entirely from timestamps and does not require any predefined schedules.

## Step 1: Import Dependencies

We need:

- `defaultdict` for grouping events
- `datetime` for timestamp processing
- `statistics` for mean and standard deviation
- `math` for rounding and confidence calculations

In [1]:
from collections import defaultdict
from datetime import datetime
import statistics
import math

## Step 2: Sample Event History

The system receives historical smart-home events.

Each event contains:

- Device ID
- Action
- Timestamp

The algorithm assumes that repeated behaviour may indicate a routine.m

In [2]:
events = [
    {"device": "living_room_light", "action": "ON", "time": "2026-06-01 18:58"},
    {"device": "living_room_light", "action": "ON", "time": "2026-06-02 19:02"},
    {"device": "living_room_light", "action": "ON", "time": "2026-06-03 19:05"},
    {"device": "living_room_light", "action": "ON", "time": "2026-06-04 19:01"},
    {"device": "living_room_light", "action": "ON", "time": "2026-06-05 19:04"},
]

## Step 3: Timestamp Parsing

Pattern extraction requires access to hour and minute values.

Therefore event timestamps are converted into Python datetime objects.

In [3]:
for e in events:
    e["time"] = datetime.strptime(
        e["time"],
        "%Y-%m-%d %H:%M"
    )

events

[{'device': 'living_room_light',
  'action': 'ON',
  'time': datetime.datetime(2026, 6, 1, 18, 58)},
 {'device': 'living_room_light',
  'action': 'ON',
  'time': datetime.datetime(2026, 6, 2, 19, 2)},
 {'device': 'living_room_light',
  'action': 'ON',
  'time': datetime.datetime(2026, 6, 3, 19, 5)},
 {'device': 'living_room_light',
  'action': 'ON',
  'time': datetime.datetime(2026, 6, 4, 19, 1)},
 {'device': 'living_room_light',
  'action': 'ON',
  'time': datetime.datetime(2026, 6, 5, 19, 4)}]

## Step 4: Convert Time into Minutes Since Midnight

Instead of comparing clock strings such as:

18:58
19:02
19:05

we transform them into numerical values:

18:58 -> 1138
19:02 -> 1142
19:05 -> 1145

This makes clustering and statistical analysis easier.

In [4]:
def minutes_of_day(ts):
    return ts.hour * 60 + ts.minute

for e in events:
    print(
        e["time"].strftime("%H:%M"),
        "->",
        minutes_of_day(e["time"])
    )

18:58 -> 1138
19:02 -> 1142
19:05 -> 1145
19:01 -> 1141
19:04 -> 1144


## Step 5: Group Similar Events

The algorithm searches for routines independently for every:

(device, action)

Examples:

(living_room_light, ON)

(water_motor, OFF)

(porch_light, ON)

Each group may contain a completely different daily routine.

In [5]:
groups = defaultdict(list)

for e in events:
    key = (e["device"], e["action"])
    groups[key].append(
        minutes_of_day(e["time"])
    )

groups

defaultdict(list,
            {('living_room_light', 'ON'): [1138, 1142, 1145, 1141, 1144]})

## Step 6: Time Bucketing

Times are divided into fixed intervals.

With a 30-minute bucket size:

18:30–18:59 -> Bucket 37

19:00–19:29 -> Bucket 38

This creates a rough histogram of when actions usually occur.

In [6]:
bucket_size = 30

minutes = groups[("living_room_light", "ON")]

bucket_counts = defaultdict(list)

for m in minutes:
    bucket_counts[m // bucket_size].append(m)

bucket_counts

defaultdict(list, {37: [1138], 38: [1142, 1145, 1141, 1144]})

## Step 7: Identify the Most Frequent Time Region

The bucket containing the highest number of events is called the dominant bucket.

This bucket becomes the candidate routine.

In [7]:
for bucket, vals in bucket_counts.items():
    print(
        f"Bucket {bucket}: {len(vals)} events"
    )

Bucket 37: 1 events
Bucket 38: 4 events


## Step 8: Select Dominant Bucket

The dominant bucket contains the largest concentration of observations.

This represents the most likely daily routine.

In [8]:
dominant_bucket = max(
    bucket_counts,
    key=lambda b: len(bucket_counts[b])
)

dominant_bucket

38

## Step 9: Bucket Center

For a 30-minute bucket:

Bucket 38

starts at:

38 × 30 = 1140

and ends at:

1169

The centre is approximately:

1155 minutes (19:15)

In [9]:
center = (
    dominant_bucket * bucket_size
    + bucket_size/2
)

center

1155.0

## Step 10: Boundary Correction

A routine may occur near a bucket boundary.

Example:

18:58
19:02

These fall into different buckets despite representing the same behaviour.

The sliding window captures nearby observations and merges them into a single cluster.

In [10]:
cluster = [
    m
    for m in minutes
    if abs(m - center) <= bucket_size
]

cluster

[1138, 1142, 1145, 1141, 1144]

## Step 11: Estimate Typical Time

The average of the clustered observations becomes the routine time.

Example:

1138
1142
1145

Average:

1141.67

≈ 19:02

In [11]:
mean_time = statistics.mean(cluster)

mean_time

1142

## Step 12: Measure Consistency

Standard deviation measures how tightly observations are grouped.

Low deviation:

19:00
19:01
19:02

indicates a very stable routine.

High deviation indicates irregular behaviour.

In [12]:
stddev = statistics.pstdev(cluster)

stddev

2.449489742783178

## Step 13: Support Score

Support measures frequency.

A pattern observed many times is more trustworthy than a pattern observed only a few times.

In [13]:
occurrences = len(cluster)

support_score = min(
    occurrences / 30,
    1.0
)

support_score

0.16666666666666666

## Step 14: Consistency Score

Consistency rewards low variance.

Small standard deviation

→ high consistency

Large standard deviation

→ lower consistency

In [14]:
consistency_score = max(
    0,
    1 - (stddev / bucket_size)
)

consistency_score

0.9183503419072274

## Step 15: Pattern Confidence

The final confidence combines:

- Support
- Consistency

High confidence means:

- The behaviour happens frequently
- The timing is predictable

In [15]:
confidence = (
    support_score +
    consistency_score
) / 2

confidence

0.5425085042869471

## Step 16: Convert Back to Clock Time

Humans understand:

19:02

better than:

1141.67 minutes

In [16]:
def fmt_hhmm(total_minutes):
    m = int(round(total_minutes))
    return f"{m//60:02d}:{m%60:02d}"

fmt_hhmm(mean_time)

'19:02'

## Step 17: Generated Routine

Final extracted pattern:

Living room light usually turns ON around 19:02.

This pattern can later be used for:

- Automation
- Anomaly detection
- Predictions
- Context-aware smart home intelligence

In [17]:
pattern = {
    "pattern_id": "TIME#living_room_light#ON",
    "usual_time": fmt_hhmm(mean_time),
    "window_minutes": max(
        bucket_size,
        math.ceil(stddev)
    ),
    "occurrences": occurrences,
    "confidence": round(confidence, 3)
}

pattern

{'pattern_id': 'TIME#living_room_light#ON',
 'usual_time': '19:02',
 'window_minutes': 30,
 'occurrences': 5,
 'confidence': 0.543}

# Limitations

## Midnight Wraparound Problem

23:58 and 00:02 are only 4 minutes apart.

However:

1438 - 2 = 1436

The current algorithm incorrectly treats them as far apart.

A circular time representation should be used.

---

## Multiple Daily Routines

A device may turn ON:

09:00
19:00

every day.

Current implementation keeps only the dominant cluster.

Future versions could detect multiple routine clusters.

---

## Fixed Bucket Size

A 30-minute bucket may be too coarse or too fine depending on the device.

Adaptive clustering methods such as DBSCAN or Gaussian Mixture Models could improve accuracy.